In [ ]:
import sys
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

from tqdm.auto import tqdm

import plotly
import kaleido

In [ ]:
#REQUIRED FUNCTIONS

In [ ]:
def plot_data_single_panel_average(mpsat_results, initial_db, true_type, predicted_types, panel_label_symbol):
    """
    Plot a single panel with one averaged spectrum for spectra matching
    the specified true and predicted polymer classes.

    Parameters:
        mpsat_results (pd.DataFrame): Table with columns "true", "predicted", and "File name".
        initial_db (pd.DataFrame): Spectral database. Spectral columns are assumed to start from column index 7.
        true_type (str): True polymer class, e.g. 'PE', 'PP', 'PS'.
        predicted_types (str): Predicted polymer class/classes, e.g. 'PE' or 'PE+PP'.
        panel_label_symbol (str): Panel label shown in the upper-left corner and used in the filename.
    """

    # A4 paper dimensions in inches: (width=8.3, height is reduced below)
    a4_width_inches = 8.3
    reduced_height_inches = 3.0  # Adjust height for a single panel

    # Wavenumber positions for vertical lines
    wavenumbers = {
        'PE' : [2915, 2845, 1472, 1467, 1377, 1462, 730, 717],
        'PA' : [687, 1199, 1274, 1372, 1464, 1538, 1634, 2858, 2932, 3298],
        'PP' : [2950, 2915, 2838, 1455, 1377, 1166, 997, 972, 840, 808],
        'PS' : [3024, 2847, 1601, 1492, 1451, 1027, 694, 537],
        'PVC': [616, 966, 1099, 1255, 1331, 1427],
        'PU' : [1223, 1451, 1531, 1731, 2865],
        'EVA': [720, 1020, 1241, 1469, 1740, 2848, 2917],
        'CPE': [2965, 2926, 2854, 1468, 1260, 653],
        'PC' : [828, 1013, 1158, 1186, 1364, 1409, 1503, 1768, 2966],
        'CA' : [600, 904, 1368, 1743],
        'PET': [720, 1094, 1241, 1713],
        ''   : []
    }

    # Create a figure with a single subplot
    fig, ax = plt.subplots(1, 1, figsize=(a4_width_inches, reduced_height_inches))

    # Panel label
    panel_label = panel_label_symbol

    # Filter data for the current true/predicted combination.
    # Important: use mpsat_results here, not a global df variable.
    list_ = mpsat_results.loc[
        (mpsat_results["true"] == true_type) &
        (mpsat_results["predicted"] == predicted_types),
        "File name"
    ].to_list()

    selected = initial_db[initial_db["File name"].isin(list_)]

    if selected.empty:
        raise ValueError(
            f"No spectra found for true={true_type!r}, predicted={predicted_types!r}."
        )

    # Extract x-axis values from column names (assuming data starts from column index 7)
    x_values = np.array([float(col) for col in selected.columns[7:]])

    # Prepare y-axis values and calculate the averaged spectrum
    spectra = selected.iloc[:, 7:].astype(float)
    average_spectrum = spectra.mean(axis=0).values

    # Plot one averaged spectrum only
    ax.plot(x_values, average_spectrum, color='black', linewidth=1.8)  # Black outline
    ax.plot(x_values, average_spectrum, color='gray', linewidth=1.2)   # Inner color

    # Add vertical lines at characteristic wavenumbers
    polymers_for_peaks = [true_type] + [item for item in str(predicted_types).split('+') if item not in [true_type]]
    colors = ['red', 'blue', 'green']

    for pol, col in zip(polymers_for_peaks, colors):
        for wn in wavenumbers.get(pol, []):
            ax.axvline(x=wn, color=col, linestyle='--', linewidth=1.5, alpha=0.5)

    # Set labels and limits
    ax.set_ylabel("Absorbance, a.u.", fontsize=12, fontweight='bold')
    # ax.set_ylim(-0.02, 0.35)
    ax.tick_params(axis='y', labelsize=14)

    # Add panel label
    ax.text(0.01, 0.9, panel_label, transform=ax.transAxes, fontsize=14, fontweight='bold')

    legend_lines = []
    legend_line_1 = mlines.Line2D([], [], color='gray', linewidth=2, label=true_type + ' spectrum (av.)')
    legend_lines.append(legend_line_1)

    for pol, col in zip(polymers_for_peaks, colors):
        legend_line_ = mlines.Line2D([], [], color=col, linewidth=2, label=pol + ' lines')
        legend_lines.append(legend_line_)

    # Add all legend lines in a single call to ax.legend()
    ax.legend(handles=legend_lines,
              loc='upper right',
              fontsize=10,
              frameon=False,
              bbox_to_anchor=(1.01, 1),
              borderaxespad=0.5,
              ncol=1)

    # Add x-axis label
    ax.set_xlabel("Wavenumber, cm$^{-1}$", fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', labelsize=14)
    ax.set_xlim(450, 4000)

    # Adjust layout for resized A4 dimensions and spacing
    plt.tight_layout()

    # Save the figure as a PNG file.
    # Same filename as in plot_data_single_panel, but with the prefix "average_".
    plt.savefig(f"average_{panel_label_symbol}_{true_type}={predicted_types}_absorbance.png",
                dpi=600,
                bbox_inches='tight')

    # Show the plot
    plt.show()

In [ ]:
#END REQUIRED FUNCTIONS

In [ ]:
#MAIN

In [ ]:
red_db_file_baseline  = "../../4_baseline_correction/MICROSCAN_database_baseline_corrected.csv"
red_db_baseline = pd.read_csv(red_db_file_baseline)
red_db_baseline.head()

In [ ]:
red_db_file  = "../../2_compiling_unified_database/MICROSCAN_database.csv"
red_db = pd.read_csv(red_db_file)
red_db.head()

In [ ]:
# Load CNN1D predictions
df = pd.read_csv('../../3_applying_CNN1D/manual_and_CNN1D_classification.csv')
df.head()

In [ ]:
#DRAW

In [ ]:
# Load CNN1D results
with open('../../3_applying_CNN1D/cnn1d_results_full_db.pickle', 'rb') as file:
    results_red = pickle.load(file)

In [ ]:
def evaluate(true_, predicted_, panel_label_symbol_):
    filenames   = df.loc[(df["true"] == true_) & (df["predicted"] == predicted_), "File name"].to_list()
    for i in filenames:
        print(i, results_red[i][0:3])

    plot_data_single_panel_average(mpsat_results=df, initial_db=red_db_baseline, true_type=true_, predicted_types=predicted_, panel_label_symbol=panel_label_symbol_)

In [ ]:
for true_class in ['PE', 'PP', 'PS']:
    unique_predicted_values = df.loc[df["true"] == true_class, "predicted"].unique()
    for pred_class in tqdm(unique_predicted_values):
        if pred_class == 'PE+PA':
            evaluate(true_=true_class, predicted_=pred_class, panel_label_symbol_='')
        elif pred_class == 'PE+PVC':
            evaluate(true_=true_class, predicted_=pred_class, panel_label_symbol_='')
        elif pred_class == 'PE+CPE':
            evaluate(true_=true_class, predicted_=pred_class, panel_label_symbol_='')  